In [0]:
import os
import sqlite3
import shutil
import tempfile
import json
import logging
import pandas as pd
from datetime import datetime
from huggingface_hub import HfApi, hf_hub_download
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
REPO_ID = "leonardoblas/us_election_2024_telegram_distilled"

BASE_DIR = "/Volumes/workspace/default/telegram_project"
RAW_PARQUET_DIR = f"{BASE_DIR}/processed/all_channels_raw"
CHECKPOINT_PATH = f"{BASE_DIR}/processed/ingestion_checkpoint.json"
LOG_PATH = f"{BASE_DIR}/ingestion.log"

os.makedirs(RAW_PARQUET_DIR, exist_ok=True)

logger = logging.getLogger("telegram_ingest")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_PATH, mode="a")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

print(f"Log file: {LOG_PATH}")

In [0]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            return json.load(f)
    return {"completed_folders": [], "stats": {"total_seen": 0, "total_kept": 0, "total_errors": 0}}

def save_checkpoint(ckpt):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(ckpt, f, indent=2)

In [0]:
def safe_int_series(s, default=0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype("Int64")

def safe_float_series(s, default=0.0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype(float)

def safe_str_series(s, default=""):
    return s.fillna(default).astype(str)

INT_COLS = [
    "message_id", "edit_hide", "noforwards", "scheduled", "pinned",
    "views", "forwards", "replies", "ttl", "forward", "reply",
    "reply_scheduled", "chain_imported", "media_price",
    "web_preview", "story", "photo", "video", "round", "voice",
    "other_media", "media_ttl", "buttons", "political"
]

FLOAT_COLS = [
    "toxicity", "severe_toxicity", "identity_attack",
    "insult", "profanity", "threat"
]

STR_COLS = [
    "content", "language", "from_id", "post_author", "date", "edit_date",
    "reply_source", "chain_from_id", "chain_from_name", "chain_post_author",
    "chain_date", "invoice", "contact", "location", "emails", "phones",
    "hashtags", "cashtags", "cards", "urls", "cleaner_urls",
    "domains", "identifiers", "restriction_reasons"
]

ALL_COLS = list(set(INT_COLS + FLOAT_COLS + STR_COLS))


def normalize_chunk(df, channel_name, source_db):
    for col in ALL_COLS:
        if col not in df.columns:
            df[col] = None
    for col in INT_COLS:
        df[col] = safe_int_series(df[col])
    for col in FLOAT_COLS:
        df[col] = safe_float_series(df[col])
    for col in STR_COLS:
        df[col] = safe_str_series(df[col])
    df["channel_name"] = channel_name
    df["source_db"] = source_db
    return df


def filter_chunk(df):
    has_content = df["content"].str.strip().str.len() > 0
    has_network = (
        (df["forward"] > 0) | (df["forwards"] > 0) |
        (df["views"] > 0) | (df["replies"] > 0) | (df["reply"] > 0) |
        (df["chain_from_id"].str.strip().str.len() > 0) |
        (df["reply_source"].str.strip().str.len() > 0)
    )
    has_media = (
        (df["photo"] > 0) | (df["video"] > 0) |
        (df["round"] > 0) | (df["voice"] > 0) |
        (df["story"] > 0) | (df["other_media"] > 0) |
        (df["web_preview"] > 0) | (df["buttons"] > 0)
    )
    return df[has_content | has_network | has_media].copy()

In [0]:
def ingest_folder(folder_prefix, api, output_dir, part_counter_start=0):
    files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
    db_files = [f for f in files if f.startswith(folder_prefix) and f.endswith(".db")]

    logger.info(f"FOLDER {folder_prefix}: {len(db_files)} .db files")

    tmp_dir = tempfile.mkdtemp()
    part_idx = part_counter_start
    folder_seen = 0
    folder_kept = 0
    folder_errors = 0

    for file_idx, repo_file in enumerate(db_files, start=1):
        channel_name = repo_file.split("/")[0]
        db_filename = os.path.basename(repo_file)
        local_db = os.path.join(tmp_dir, db_filename)

        try:
            local_cached = hf_hub_download(
                repo_id=REPO_ID, filename=repo_file,
                repo_type="dataset", token=HF_TOKEN
            )
            shutil.copy2(local_cached, local_db)

            conn = sqlite3.connect(local_db)
            tables = pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table'", conn
            )
            if "messages" not in set(tables["name"].tolist()):
                conn.close()
                os.remove(local_db)
                continue

            for chunk_no, chunk in enumerate(
                pd.read_sql_query("SELECT * FROM messages", conn, chunksize=50000),
                start=1
            ):
                n_orig = len(chunk)
                folder_seen += n_orig
                chunk = normalize_chunk(chunk, channel_name, repo_file)
                chunk = filter_chunk(chunk)
                n_kept = len(chunk)
                folder_kept += n_kept

                if n_kept > 0:
                    part_path = os.path.join(output_dir, f"part_{part_idx:06d}.parquet")
                    chunk.to_parquet(part_path, index=False)
                    part_idx += 1

            conn.close()
            os.remove(local_db)

            if file_idx % 200 == 0:
                logger.info(f"  [{file_idx}/{len(db_files)}] seen={folder_seen} kept={folder_kept}")

        except Exception as e:
            folder_errors += 1
            logger.error(f"  ERROR {repo_file}: {e}")
            try:
                if os.path.exists(local_db):
                    os.remove(local_db)
            except:
                pass

    shutil.rmtree(tmp_dir, ignore_errors=True)
    logger.info(f"FOLDER {folder_prefix} DONE: seen={folder_seen} kept={folder_kept} errors={folder_errors}")
    return folder_seen, folder_kept, folder_errors, part_idx

In [0]:
api = HfApi(token=HF_TOKEN)
ckpt = load_checkpoint()

all_folders = [f"channels_{i}/" for i in range(10, 11)]

existing_parts = [f for f in os.listdir(RAW_PARQUET_DIR) if f.endswith(".parquet")]
part_counter = len(existing_parts)

print(f"Resume from part_{part_counter:06d} | Completed: {len(ckpt['completed_folders'])} folders")

for folder in all_folders:
    if folder in ckpt["completed_folders"]:
        continue

    seen, kept, errors, part_counter = ingest_folder(
        folder, api, RAW_PARQUET_DIR, part_counter
    )

    ckpt["completed_folders"].append(folder)
    ckpt["stats"]["total_seen"] += seen
    ckpt["stats"]["total_kept"] += kept
    ckpt["stats"]["total_errors"] += errors
    save_checkpoint(ckpt)

    print(
        f"✓ {folder:<15} seen={seen:>10,} kept={kept:>10,} err={errors:>4} | "
        f"TOTAL seen={ckpt['stats']['total_seen']:>12,} kept={ckpt['stats']['total_kept']:>12,}"
    )

print("INGESTION COMPLETE")
print(json.dumps(ckpt["stats"], indent=2))

In [0]:
with open(LOG_PATH) as f:
    lines = f.readlines()
    for line in lines[-30:]:
        print(line.strip())

In [0]:
df_all = spark.read.parquet(f"{RAW_PARQUET_DIR}/")
print(f"Total rows: {df_all.count():,}")
print(f"Distinct channels: {df_all.select('channel_name').distinct().count():,}")
display(df_all.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql import Window

df_raw = spark.read.parquet(f"{RAW_PARQUET_DIR}/")

df_clean = (df_raw
    .withColumn("timestamp", F.to_timestamp(F.col("date")))
    .withColumn("edit_timestamp", F.to_timestamp(F.col("edit_date")))
    .filter(F.col("timestamp").isNotNull())
    .withColumn("date_only", F.to_date("timestamp"))
    .withColumn("hour_of_day", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("week_number", F.weekofyear("timestamp"))
    .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
    .filter(
        (F.col("timestamp") >= "2024-07-01") &
        (F.col("timestamp") <= "2025-01-01")
    )
)

print(f"After timestamp filter: {df_clean.count():,}")

In [0]:
from pyspark.sql.functions import udf

@udf(StringType())
def clean_text(text):
    if not text or text.strip() == "":
        return ""
    import re
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r't\.me/\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    emoji_pat = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pat.sub(' ', text)
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'__(.+?)__', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = (df_clean
    .withColumn("content_clean", clean_text(F.col("content")))
    .withColumn("word_count",
        F.when(F.length("content_clean") > 0,
               F.size(F.split("content_clean", r"\s+")))
        .otherwise(0)
    )
)

In [0]:
w = Window.partitionBy("channel_name", "content_clean").orderBy("timestamp")

df_clean = (df_clean
    .withColumn("_rn", F.row_number().over(w))
    .filter((F.col("content_clean") == "") | (F.col("_rn") == 1))
    .drop("_rn")
)

print(f"After dedup: {df_clean.count():,}")

In [0]:
CLEAN_PATH = f"{BASE_DIR}/processed/messages_clean_delta"

(df_clean
    .repartition(200)
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("year_month")
    .save(CLEAN_PATH)
)

spark.sql(f"CREATE TABLE IF NOT EXISTS telegram_messages_clean USING DELTA LOCATION '{CLEAN_PATH}'")
print(f"Saved: {CLEAN_PATH}")